# KPoEM Emotion Distribution Visualization

* 본 코드는 KPoEM 데이터셋에 포함된 개별 시인의 작품을 대상으로 감정 분포를 시각화하기 위한 코드이다.
* 입력 데이터는 KPoEM 데이터셋의 감정 라벨링 결과를 기반으로 하며, 각 감정의 출현 빈도를 집계하여 시각화한다.
* 시각화 결과는 특정 시인의 작품 세계에서 두드러지게 나타나는 감정적 경향과 분포를 탐색하는 데 활용될 수 있다.
* 본 코드는 김소월, 윤동주, 이상, 임화, 한용운 등 KPoEM에 포함된 시인별 감정 분포 분석에 공통적으로 적용 가능하도록 설계되었다.
* 생성된 시각화는 KPoEM 플랫폼의 Distant Reading 기능에서 활용된다.

In [1]:
import pandas as pd
import plotly.express as px
from collections import Counter

In [2]:
url = "https://huggingface.co/datasets/AKS-DHLAB/KPoEM/resolve/main/KPoEM_line_dataset_v4.tsv"

df = pd.read_csv(
    url,
    sep="\t",
    encoding="utf-8",
    quoting=3
)

df.head()


,line_id,poem_id,text,sub_title,title,poet,annotator_01,annotator_02,annotator_03,annotator_04,annotator_05
0,1,1,죽는 날까지 하늘을 우러러,NaN,서시,윤동주,비장함,비장함,"뿌듯함, 비장함","비장함, 뿌듯함, 감동/감탄","비장함, 서러움, 슬픔"
1,2,1,"한 점 부끄럼이 없기를,",NaN,서시,윤동주,"부끄러움, 비장함","부끄러움, 비장함, 기대감, 불안/걱정, 서러움, 슬픔","깨달음, 비장함, 뿌듯함","비장함, 부끄러움, 기대감",비장함
2,3,1,잎새에 이는 바람에도,NaN,서시,윤동주,"기대감, 신기함/관심","기대감, 불안/걱정, 비장함","슬픔, 서러움, 불안/걱정, 당황/난처","비장함, 슬픔","감동/감탄, 신기함/관심, 편안/쾌적, 기대감"
3,4,1,나는 괴로워했다.,NaN,서시,윤동주,"절망, 슬픔, 패배/자기혐오","절망, 슬픔, 패배/자기혐오, 죄책감, 힘듦/지침, 비장함","당황/난처, 서러움, 죄책감, 패배/자기혐오","비장함, 슬픔, 패배/자기혐오, 절망, 힘듦/지침","슬픔, 서러움, 절망, 힘듦/지침, 패배/자기혐오"
4,5,1,별을 노래하는 마음으로,NaN,서시,윤동주,"기쁨, 신기함/관심, 즐거움/신남, 흐뭇함(귀여움/예쁨), 뿌듯함","기쁨, 뿌듯함, 감동/감탄, 슬픔, 비장함, 아껴주는, 환영/호의, 기대감","고마움, 기대감, 기쁨, 아껴주는, 흐뭇함(귀여움/예쁨)","감동/감탄, 기대감, 기쁨, 아껴주는, 행복","즐거움/신남, 기대감, 기쁨, 행복"


In [3]:
url = "https://huggingface.co/datasets/AKS-DHLAB/KPoEM/resolve/main/KPoEM_poem_dataset_v4.tsv"

df_poem = pd.read_csv(
    url,
    sep="\t",
    encoding="utf-8",
    quoting=3
)

df_poem.head(10)


,seg_id,poem_id,text,sub_title,title,poetry_book,poet,annotator_01,annotator_02,annotator_03,annotator_04,annotator_05
0,1,1,"죽는 날까지 하늘을 우러러 한 점 부끄럼이 없기를, 잎새에 이는 바람에도 나는 괴로...",NaN,서시,하늘과 바람과 별과 시,윤동주,"불안/걱정, 비장함, 서러움, 슬픔, 안타까움/실망, 부끄러움, 죄책감, 패배/자기혐오","패배/자기혐오, 서러움, 비장함, 부끄러움, 아껴주는, 슬픔, 불안/걱정, 죄책감","깨달음, 흐뭇함(귀여움/예쁨), 고마움, 힘듦/지침, 안타까움/실망","비장함, 뿌듯함, 슬픔","비장함, 감동/감탄, 깨달음, 서러움"
1,2,2,산모퉁이를 돌아 논가 외딴 우물을 홀로 찾아가선 가만히 들여다봅니다. 우물 속에는 ...,NaN,자화상,하늘과 바람과 별과 시,윤동주,"불쌍함/연민, 아껴주는, 신기함/관심, 흐뭇함(귀여움/예쁨), 서러움, 짜증, 지긋...","신기함/관심, 증오/혐오, 불쌍함/연민, 화남/분노, 서러움, 슬픔, 아껴주는, 패...","안타까움/실망, 서러움, 슬픔, 깨달음, 흐뭇함(귀여움/예쁨)","불평/불만, 안타까움/실망, 의심/불신, 슬픔","불쌍함/연민, 불안/걱정"
2,3,3,쫓아오든 햇빛인데 지금 교회당 꼭대기 십자가에 걸리었습니다. 첨탑(尖塔)이 저렇게...,NaN,십자가,하늘과 바람과 별과 시,윤동주,"서러움, 패배/자기혐오, 존경, 감동/감탄, 비장함, 불쌍함/연민, 불안/걱정, 죄책감","감동/감탄, 놀람, 불쌍함/연민, 비장함, 슬픔, 행복, 아껴주는, 불안/걱정","절망, 깨달음, 서러움, 힘듦/지침, 존경, 비장함","당황/난처, 힘듦/지침, 놀람, 슬픔, 불안/걱정","깨달음, 비장함, 존경, 신기함/관심"
3,4,4,바람이 어디로부터 불어 와 어디로 불려 가는 것일까 바람이 부는데 내 괴로움에는 ...,NaN,바람이 불어,하늘과 바람과 별과 시,윤동주,"슬픔, 서러움, 한심함, 패배/자기혐오, 죄책감, 부끄러움","비장함, 서러움, 슬픔, 불안/걱정, 패배/자기혐오, 죄책감","힘듦/지침, 안타까움/실망, 서러움, 절망, 깨달음","힘듦/지침, 당황/난처, 불안/걱정, 불평/불만, 슬픔, 부끄러움, 패배/자기혐오","깨달음, 불쌍함/연민, 불안/걱정, 서러움, 패배/자기혐오, 힘듦/지침"
4,5,5,고향에 돌아온 날 밤에 내 백골(白骨)이 따라와 한방에 누웠다. 어둔 방은 우주로...,NaN,또 다른 고향,하늘과 바람과 별과 시,윤동주,"불쌍함/연민, 비장함, 공포/무서움, 슬픔, 서러움, 의심/불신, 깨달음, 기대감","힘듦/지침, 놀람, 서러움, 슬픔, 감동/감탄, 존경, 죄책감, 패배/자기혐오, 비...","절망, 깨달음, 서러움, 힘듦/지침, 안타까움/실망","슬픔, 힘듦/지침, 절망, 기대감, 안타까움/실망, 불안/걱정","공포/무서움, 당황/난처, 놀람, 불안/걱정, 비장함"
5,6,6,계절이 지나가는 하늘에는 가을로 가득 차 있습니다. 나는 아무 걱정도 없이 가을 ...,NaN,별 헤는 밤,하늘과 바람과 별과 시,윤동주,"흐뭇함(귀여움/예쁨), 감동/감탄, 아껴주는, 슬픔, 기쁨, 기대감, 깨달음, 환영...","감동/감탄, 기대감, 서러움, 슬픔, 비장함, 아껴주는, 흐뭇함(귀여움/예쁨), 안...","흐뭇함(귀여움/예쁨), 기대감, 안타까움/실망, 서러움, 슬픔","감동/감탄, 기쁨, 흐뭇함(귀여움/예쁨), 행복, 아껴주는, 편안/쾌적","존경, 감동/감탄, 신기함/관심, 깨달음, 서러움, 슬픔, 안타까움/실망"
6,7,6,"어머님, 나는 별 하나에 아름다운 말 한 마디씩 불러 봅니다. 소학교 때 책상을 같...",NaN,별 헤는 밤,하늘과 바람과 별과 시,윤동주,"아껴주는, 슬픔, 부끄러움, 불쌍함/연민, 존경, 감동/감탄, 고마움, 흐뭇함(귀여...","아껴주는, 흐뭇함(귀여움/예쁨), 환영/호의, 서러움, 슬픔, 깨달음, 죄책감, 부...","안타까움/실망, 서러움, 기대감, 슬픔, 깨달음","불안/걱정, 서러움, 슬픔, 존경, 환영/호의","서러움, 슬픔, 존경"
7,8,7,"살구나무 그늘로 얼굴을 가리고, 병원 뒤뜰에 누워, 젊은 여자가 흰 옷 아래로 하얀...",NaN,병원,하늘과 바람과 별과 시,윤동주,"불쌍함/연민, 슬픔, 서러움, 안타까움/실망, 힘듦/지침, 패배/자기혐오, 부담/안...","신기함/관심, 불쌍함/연민, 서러움, 슬픔, 안타까움/실망, 당황/난처, 힘듦/지침...","안타까움/실망, 서러움, 힘듦/지침, 고마움, 기대감","서러움, 불안/걱정, 불평/불만, 화남/분노, 의심/불신, 증오/혐오","신기함/관심, 불안/걱정, 불평/불만, 불쌍함/연민"
8,9,8,잃어버렸습니다. 무얼 어디다 잃었는지 몰라 두 손의 호주머니를 더듬어 길에 나...,NaN,길,하늘과 바람과 별과 시,윤동주,"당황/난처, 슬픔, 안타까움/실망, 부끄러움, 부담/안_내킴, 깨달음, 패배/자기혐...","당황/난처, 불안/걱정, 놀람, 서러움, 슬픔, 부끄러움, 감동/감탄, 죄책감, 깨...","안타까움/실망, 서러움, 슬픔, 깨달음, 힘듦/지침","당황/난처, 불안/걱정, 슬픔, 힘듦/지침","불안/걱정, 비장함, 당황/난처, 깨달음"
9,10,9,세상으로부터 돌아오듯이 이제 내 좁은방에 돌아와 불을 끄옵니다. 불을 켜 두는 것은...,NaN,돌아와 보는 밤,하늘과 바람과 별과 시,윤동주,"힘듦/지침, 서러움, 기대감, 깨달음, 슬픔, 비장함","힘듦/지침, 불안/걱정, 부담/안_내킴, 공포/무서움, 불쌍함/연민, 서러움, 슬픔...","힘듦/지침, 깨달음, 편안/쾌적, 슬픔, 불안/걱정","서러움, 슬픔, 힘듦/지침, 불안/걱정","힘듦/지침, 서러움, 깨달음"


## Generate English emotion-distribution charts for five poets


In [ ]:
# English KOTE 44 labels used by the distribution charts
KOTE_44 = [
    "Complaint / Dissatisfaction", "Welcome / Favor", "Impressed / Admiration",
    "Fed up", "Gratitude", "Sadness", "Anger / Rage", "Respect", "Anticipation",
    "Arrogance / Disregard", "Pitifulness / Disappointment", "Resolute",
    "Doubt / Distrust", "Pride", "Comfort / Cozy", "Curiosity / Interest", "Caring",
    "Shame", "Fear/Scary", "Despair", "Pathetic ", "Disgust / Repulsiveness",
    "Irritation", "Preposterous ", "NO EMOTION", "Defeat / Self-hatred", "Laziness",
    "Fatigue / Exhaustion", "Pleasure / Excitement", "Realization", "Guilt",
    "Loathing / Hatred", "Pleased (Cute / Pretty)", "Embarrassment / Awkwardness",
    "Shock", "Burden / Unwillingness", "Sorrow", "Boredom", "Pity / Compassion",
    "Surprise", "Happiness", "Anxiety / Worry", "Joy", "Relief / Trust",
]

KOTE_KO = [
    '불평/불만','환영/호의','감동/감탄','지긋지긋','고마움','슬픔','화남/분노','존경','기대감',
    '우쭐댐/무시함','안타까움/실망','비장함','의심/불신','뿌듯함','편안/쾌적','신기함/관심','아껴주는',
    '부끄러움','공포/무서움','절망','한심함','역겨움/징그러움','짜증','어이없음','없음','패배/자기혐오',
    '귀찮음','힘듦/지침','즐거움/신남','깨달음','죄책감','증오/혐오','흐뭇함(귀여움/예쁨)','당황/난처',
    '경악','부담/안_내킴','서러움','재미없음','불쌍함/연민','놀람','행복','불안/걱정','기쁨','안심/신뢰'
]

KOTE_EN = dict(zip(KOTE_KO, KOTE_44))
POETS = {
    "김소월": ("Kim Sowol", "kim", "kim_sowol"),
    "윤동주": ("Yun Dong-ju", "yun", "yun_dongju"),
    "이상": ("Yi Sang", "lee", "yi_sang"),
    "임화": ("Im Hwa", "lim", "im_hwa"),
    "한용운": ("Han Yong-un", "han", "han_yongun"),
}

from pathlib import Path
from collections import Counter
import pandas as pd
import plotly.express as px

repo_root = Path.cwd()
if repo_root.name == "backend":
    repo_root = repo_root.parent
output_dir = repo_root / "backend" / "english_analysis_html"
output_dir.mkdir(parents=True, exist_ok=True)

annotator_columns = [f"annotator_0{i}" for i in range(1, 6)]
df_all = pd.concat([df, df_poem], ignore_index=True)

for poet_ko, (poet_en, source_dir, slug) in POETS.items():
    poet_data = df_all[df_all["poet"] == poet_ko]
    counts = Counter()
    for value in poet_data[annotator_columns].to_numpy().ravel():
        if pd.notna(value):
            counts.update(item.strip() for item in str(value).split(",") if item.strip())

    emotion_df = pd.DataFrame({
        "Emotion": KOTE_44,
        "Count": [counts[emotion] for emotion in KOTE_KO],
    }).sort_values(["Count", "Emotion"], ascending=[False, True])

    fig = px.bar(
        emotion_df, x="Emotion", y="Count", color="Count",
        color_continuous_scale=px.colors.sequential.Plasma,
        title=f"KPoEM — Emotion Distribution: {poet_en}",
    )
    fig.update_layout(
        xaxis_title="KOTE Emotion", yaxis_title="Annotation Count",
        autosize=True, height=800, margin=dict(l=90, r=80, t=100, b=230),
        font=dict(family="Arial, sans-serif", size=13), title=dict(x=0.5),
    )
    fig.update_xaxes(tickangle=50, automargin=True)

    standalone = output_dir / f"{slug}_emotion_distribution.html"
    source = repo_root / "frontend" / "source" / source_dir / f"line_{source_dir}.html"
    fig.write_html(standalone, include_plotlyjs=True, full_html=True)
    fig.write_html(source, include_plotlyjs=True, full_html=True)
    print(f"Saved {poet_en}: {standalone} and {source}")
